In [ ]:
import sys
import os
import mlflow.xgboost
import pandas as pd

#adding the scripts frolder

sys.path.append(os.path.abspath("..")) # add the project root
from scripts.processing import extract_features

# setup MLflow

MLRUNS_DIR = "/Users/nicholus/Documents/GitHub/credit-card-default/notebooks/mlruns"
mlflow.set_tracking_uri(f"file://{MLRUNS_DIR}")

#Model details

MODEL_NAME = "CreditDefault_XGB"
MODEL_VERSION = 3



def predict_credit_risk(raw_data_dict, model_name, model_version):
    """
    Takes a raw dictionary, processes features, and returns a risk score.
    """
    try:
        # 1. Convert dict to DataFrame
        df_raw = pd.DataFrame([raw_data_dict])
        
        # 2. Extract engineered features using your module
        df_featured = extract_features(df_raw)
        
        # 3. Clean for XGBoost (Drop ID and target if they exist)
        # We also ensure all columns are floats for MLflow schema stability
        cols_to_drop = ['ID', 'default'] 
        X_inference = df_featured.drop(columns=cols_to_drop, errors='ignore').astype(float)
        
        # 4. Load the model from the Registry
        model_uri = f"models:/{model_name}/{model_version}"
        model = mlflow.xgboost.load_model(model_uri)
        
        # 5. Get Probability
        prob = model.predict_proba(X_inference)[:, 1][0]
        
        return {
            "risk_score": round(float(prob), 4),
            "decision": "REJECT" if prob > 0.5 else "APPROVE",
            "status": "success"
        }

    except Exception as e:
        return {
            "status": "error",
            "message": str(e)
        }



# Test Case 
customer_a = {
    "LIMIT_BAL": 120000, "SEX": 2, "EDUCATION": 2, "MARRIAGE": 2, "AGE": 26,
    "PAY_0": -1, "PAY_2": 2, "PAY_3": 0, "PAY_4": 0, "PAY_5": 0, "PAY_6": 2,
    "BILL_AMT1": 2682, "BILL_AMT2": 1725, "BILL_AMT3": 2682, 
    "BILL_AMT4": 3272, "BILL_AMT5": 3455, "BILL_AMT6": 3261,
    "PAY_AMT1": 0, "PAY_AMT2": 1000, "PAY_AMT3": 1000, 
    "PAY_AMT4": 1000, "PAY_AMT5": 0, "PAY_AMT6": 2000
}

result = predict_credit_risk(customer_a, "CreditDefault_XGB", 3)
print(result)